# Retrieval Augmented Generation
## Step 4.2: Retrieval

Retrieval is the centerpiece of our retrieval augmented generation (RAG) flow. It follows the steps:
- Load the embedded database
- Similarity search
- Reranker

<img src="img/RAG.JPG" width="2000"/>

### Database reload from embedded


In [ ]:
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")    

db_faiss = FAISS.load_local(
    "docs/faiss",
    embedding_model,
    allow_dangerous_deserialization=True
)

### Similarity Search

In [2]:
query = "Where most robberies occured in Chicago"

In [5]:
docs_and_scores = db_faiss.similarity_search_with_score(query, k=5)

In [12]:
for doc, score in docs_and_scores:
    print(score)
    print(doc.page_content)
    

0.8996592
L, et al. (2022) Correction: Psychologi cal impacts
from COVID-19 among university students: Risk
factors across seven states in the United States.
PLoS ONE 17(8): e0273938. https://doi.o rg/
10.1371/ journal.pone. 0273938
Published: August 26, 2022
Copyright: © 2022 Browning et al. This is an open
access article distributed under the terms of the
Creative Commons Attribution License, which
permits unrestricte d use, distribu tion, and
reproduction in any medium, provided the original
author and source are credited.
0.9349855
East and South Mexico City) with a high intensity of crashes, but not a high crime intensity.
https://d oi.org/10.1371 /journal.pone. 0246714.g002
PLOS ONE
The heartbeat of the city
PLOS ONE | https://doi.or g/10.137 1/journal.po ne.02467 14 February 24, 2021 11 / 30
0.93662155
across space [59]. Data from Chicago showed that high schools appear to attract robberies
around school hours, but that street robbers seem to perpetrate in transit hubs most of t

### Reranker

In [7]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
pairs = [(query, d[0].page_content) for d in docs_and_scores]
scores = reranker.predict(pairs)

reranked_docs = [
    doc for _, doc in sorted(
        zip(scores, docs_and_scores),
        key=lambda x: x[0],
        reverse=True
    )
]

top_docs = reranked_docs


In [10]:
for i,doc in enumerate(top_docs):
    print("reranking", i)
    print(doc[0].page_content)
    print("oldscore", doc[1])

reranking 0
across space [59]. Data from Chicago showed that high schools appear to attract robberies
around school hours, but that street robbers seem to perpetrate in transit hubs most of the
time irrespective of how many potential victims are around [60]. Data from Campinas, Brazil
was used to show that crime exhibits spatio-temporal patterns, which vary according to differ-
ent types of crime [61] and data from Floriano ´ polis, also in Brazil, was used to show that street
robberies’ hotspots change across space depending on the time of the day [62]. Even a plot
triplet made of a hotspot map, a yearly trend -with weekly data- and a daily trend -with hourly
data- was suggested as a tool to analyse crime patterns [42].
One of the key drivers of why high schools attract robberies in Chicago, or why street rob-
beries shift depending on the time of the day in Floriano ´ polis, is due to daily commutes. In
Vancouver, for example, some suburbs double their nighttime population whilst oth